# Legacy Exploratory Notebook

This notebook is preserved as a historical reference for early LDA-over-HSI experiments.

- It is not maintained as production code.
- Notebook outputs and machine-specific artifacts were removed for repository hygiene.
- Visible headings, comments, and messages were normalized to English for readability.


In [ ]:
# Legacy bootstrap hint for ad hoc notebook execution
# !pip install spectral common_utils pyLDAvis


In [ ]:
from platform import python_version

python_version()

In [ ]:
# Optional legacy dependency hints
# !pip install --upgrade gensim==4.2.0
# !pip install nltk
# !pip install wordcloud


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter

import sys, os

import numpy as np

sys.path.append(os.path.join('..'))

from argparse import ArgumentDefaultsHelpFormatter, ArgumentParser
from numpy import argsort, bincount, set_printoptions, where, zeros


In [ ]:
# Use word stemming
use_stemmer = False

# Use n-grams
use_ngrams = True
words_per_ngram = 2


In [ ]:
import gensim
gensim.__version__

In [ ]:
import json, re
import pandas as pd 
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.tokenize import ToktokTokenizer

In [ ]:
nltk.download('stopwords')

In [ ]:

from gensim.corpora import Dictionary
from gensim.models import LdaModel
import random
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
%matplotlib inline

# Helper functions


In [ ]:
def clean_text(text):
    # Remove accents from Spanish text.
    text = re.sub(r'?', 'a', str(text))
    text = re.sub(r'?', 'e', str(text))
    text = re.sub(r'?', 'i', str(text))
    text = re.sub(r'?', 'o', str(text))
    text = re.sub(r'?', 'u', str(text))

    # Remove special characters.
    text = re.sub(r'/', '', str(text))
    text = re.sub(r'\W', ' ', str(text))
    # Remove isolated single-character tokens.
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    # Collapse repeated whitespace.
    text = re.sub(r'\s+', ' ', text, flags=re.I)
    # Lowercase the normalized text.
    text = text.lower()

    # Additional heuristic fixes for the original Spanish corpus.
    # Singularization heuristics.
    text = re.sub(r'as ', 'a ', str(text))
    text = re.sub(r'es ', 'e ', str(text))
    text = re.sub(r'is ', 'i ', str(text))
    text = re.sub(r'os ', 'o ', str(text))
    text = re.sub(r'us ', 'u ', str(text))    

    # Manual normalization rules.
    text = re.sub(r'condicione ', 'condicion ', str(text))    

    text = re.sub(r'mantienen ', 'mantener ', str(text))
    text = re.sub(r'mantiene ', 'mantener ', str(text))
    text = re.sub(r'manteniendo', 'mantener', str(text))
    text = re.sub(r'mantendremos', 'mantener', str(text))
    text = re.sub(r'mantendremo', 'mantener', str(text))

    text = re.sub(r'ajustandos', 'ajustar ', str(text))
    text = re.sub(r'ajustados', 'ajustar ', str(text))    
    text = re.sub(r'ajustando', 'ajustar ', str(text))
    text = re.sub(r'ajustado', 'ajustar ', str(text))        
    text = re.sub(r'ajustan ', 'ajustar ', str(text))    
    text = re.sub(r'ajusta ', 'ajustar ', str(text))        
    text = re.sub(r'ajustarse ', 'ajustar ', str(text))    
    text = re.sub(r'ajsutarse ', 'ajustar ', str(text))    

    text = re.sub(r'metalurgicos ', 'metalurgico ', str(text))        

    text = re.sub(r'alta ', 'alto ', str(text))
    text = re.sub(r'baja ', 'bajo ', str(text))    

    text = re.sub(r'estamo ', 'estar ', str(text))        

    return text

STOPWORDS = set(stopwords.words("spanish"))

def filter_stopwords_and_digits(tokens):
    """Remove stopwords and standalone numbers from a token list."""
    return [token for token in tokens if token not in STOPWORDS 
            and not token.isdigit()]
stemmer = SnowballStemmer("spanish")

def stem_words(tokens):
    """Reduce each token in a list to its stem."""
    return [stemmer.stem(token) for token in tokens]


In [ ]:
def generate_ngrams(words, words_to_combine):
    output = []
    for i in range(len(words) - words_to_combine + 1):
        output.append("_".join(words[i:i + words_to_combine]))
    return output


In [ ]:
from os import listdir
 
def list_files(directory, extension):
    return [f for f in listdir(directory) if f.endswith('.' + extension)]

def list_files_with_prefix(directory, initname, extension):
    return [f for f in listdir(directory) if f.startswith(initname) and f.endswith('.' + extension)]


In [ ]:
# Example dataset bootstrap
# if not os.path.exists('./X_W01c.npz'):
#     !wget https://www.dropbox.com/s/0nbzaafjpb7ayt9/npz_spectra_example.zip
#     !unzip npz_spectra_example.zip


In [ ]:
# Dataset used for training and validation
folder_db = './DB/Example/'
files_x = list_files_with_prefix(folder_db, 'X', 'npz')
files_x = np.sort(np.array((files_x)))

files_x


In [ ]:
# Compute the global normalization range.
min_db = 1000000000.0
max_db = -1000000000.0
for idxfile in range(len(files_x)):
    # Load X data with shape [num_samples, num_spectra].
    filenpz = np.load(folder_db + files_x[idxfile])
    X_data = filenpz['X']
    filenpz.close()

    minf = min(np.ravel(X_data))
    maxf = max(np.ravel(X_data))
    if min_db > minf:
        min_db = minf
    if max_db < maxf:
        max_db = maxf

global_range = max_db - min_db if max_db > min_db else 1.0
print('Global min value: ' + str(min_db))
print('Global max value: ' + str(max_db))


In [ ]:
wavelength = [400.11,
              400.72,
              401.33,
            401.94,
            402.54,
            403.15,
            403.76,
            404.37,
            404.98,
            405.58,
            406.19,
            406.8,
            407.41,
            408.02,
            408.62,
            409.23,
            409.84,
            410.45,
            411.06,
            411.67,
            412.27,
            412.88,
            413.49,
            414.1,
            414.71,
            415.32,
            415.93,
            416.54,
            417.14,
            417.75,
            418.36,
            418.97,
            419.58,
            420.19,
            420.8,
            421.41,
            422.02,
            422.63,
            423.24,
            423.85,
            424.46,
            425.07,
            425.68,
            426.29,
            426.9,
            427.51,
            428.12,
            428.73,
            429.34,
            429.95,
            430.56,
            431.17,
            431.78,
            432.39,
            433.0,
            433.61,
            434.22,
            434.83,
            435.45,
            436.06,
            436.67,
            437.28,
            437.89,
            438.5,
            439.11,
            439.72,
            440.34,
            440.95,
            441.56,
            442.17,
            442.78,
            443.39,
            444.01,
            444.62,
            445.23,
            445.84,
            446.45,
            447.07,
            447.68,
            448.29,
            448.9,
            449.51,
            450.13,
            450.74,
            451.35,
            451.96,
            452.58,
            453.19,
            453.8,
            454.42,
            455.03,
            455.64,
            456.25,
            456.87,
            457.48,
            458.09,
            458.71,
            459.32,
            459.93,
            460.55,
            461.16,
            461.77,
            462.39,
            463.0,
            463.62,
            464.23,
            464.84,
            465.46,
            466.07,
            466.69,
            467.3,
            467.91,
            468.53,
            469.14,
            469.76,
            470.37,
            470.99,
            471.6,
            472.22,
            472.83,
            473.45,
            474.06,
            474.68,
            475.29,
            475.91,
            476.52,
            477.14,
            477.75,
            478.37,
            478.98,
            479.6,
            480.21,
            480.83,
            481.44,
            482.06,
            482.68,
            483.29,
            483.91,
            484.52,
            485.14,
            485.76,
            486.37,
            486.99,
            487.6,
            488.22,
            488.84,
            489.45,
            490.07,
            490.69,
            491.3,
            491.92,
            492.54,
            493.15,
            493.77,
            494.39,
            495.0,
            495.62,
            496.24,
            496.86,
            497.47,
            498.09,
            498.71,
            499.33,
            499.94,
            500.56,
            501.18,
            501.8,
            502.41,
            503.03,
            503.65,
            504.27,
            504.89,
            505.5,
            506.12,
            506.74,
            507.36,
            507.98,
            508.59,
            509.21,
            509.83,
            510.45,
            511.07,
            511.69,
            512.31,
            512.93,
            513.54,
            514.16,
            514.78,
            515.4,
            516.02,
            516.64,
            517.26,
            517.88,
            518.5,
            519.12,
            519.74,
            520.36,
            520.98,
            521.6,
            522.22,
            522.84,
            523.45,
            524.07,
            524.69,
            525.32,
            525.94,
            526.56,
            527.18,
            527.8,
            528.42,
            529.04,
            529.66,
            530.28,
            530.9,
            531.52,
            532.14,
            532.76,
            533.38,
            534.0,
            534.62,
            535.25,
            535.87,
            536.49,
            537.11,
            537.73,
            538.35,
            538.97,
            539.59,
            540.22,
            540.84,
            541.46,
            542.08,
            542.7,
            543.33,
            543.95,
            544.57,
            545.19,
            545.81,
            546.44,
            547.06,
            547.68,
            548.3,
            548.93,
            549.55,
            550.17,
            550.79,
            551.42,
            552.04,
            552.66,
            553.28,
            553.91,
            554.53,
            555.15,
            555.78,
            556.4,
            557.02,
            557.65,
            558.27,
            558.89,
            559.52,
            560.14,
            560.76,
            561.39,
            562.01,
            562.64,
            563.26,
            563.88,
            564.51,
            565.13,
            565.76,
            566.38,
            567.0,
            567.63,
            568.25,
            568.88,
            569.5,
            570.13,
            570.75,
            571.38,
            572.0,
            572.63,
            573.25,
            573.88,
            574.5,
            575.13,
            575.75,
            576.38,
            577.0,
            577.63,
            578.25,
            578.88,
            579.5,
            580.13,
            580.75,
            581.38,
            582.01,
            582.63,
            583.26,
            583.88,
            584.51,
            585.14,
            585.76,
            586.39,
            587.02,
            587.64,
            588.27,
            588.89,
            589.52,
            590.15,
            590.77,
            591.4,
            592.03,
            592.65,
            593.28,
            593.91,
            594.54,
            595.16,
            595.79,
            596.42,
            597.04,
            597.67,
            598.3,
            598.93,
            599.55,
            600.18,
            600.81,
            601.44,
            602.07,
            602.69,
            603.32,
            603.95,
            604.58,
            605.21,
            605.83,
            606.46,
            607.09,
            607.72,
            608.35,
            608.98,
            609.6,
            610.23,
            610.86,
            611.49,
            612.12,
            612.75,
            613.38,
            614.01,
            614.63,
            615.26,
            615.89,
            616.52,
            617.15,
            617.78,
            618.41,
            619.04,
            619.67,
            620.3,
            620.93,
            621.56,
            622.19,
            622.82,
            623.45,
            624.08,
            624.71,
            625.34,
            625.97,
            626.6,
            627.23,
            627.86,
            628.49,
            629.12,
            629.75,
            630.38,
            631.01,
            631.64,
            632.27,
            632.91,
            633.54,
            634.17,
            634.8,
            635.43,
            636.06,
            636.69,
            637.32,
            637.95,
            638.59,
            639.22,
            639.85,
            640.48,
            641.11,
            641.74,
            642.38,
            643.01,
            643.64,
            644.27,
            644.9,
            645.54,
            646.17,
            646.8,
            647.43,
            648.07,
            648.7,
            649.33,
            649.96,
            650.6,
            651.23,
            651.86,
            652.49,
            653.13,
            653.76,
            654.39,
            655.03,
            655.66,
            656.29,
            656.93,
            657.56,
            658.19,
            658.83,
            659.46,
            660.09,
            660.73,
            661.36,
            661.99,
            662.63,
            663.26,
            663.9,
            664.53,
            665.16,
            665.8,
            666.43,
            667.07,
            667.7,
            668.34,
            668.97,
            669.6,
            670.24,
            670.87,
            671.51,
            672.14,
            672.78,
            673.41,
            674.05,
            674.68,
            675.32,
            675.95,
            676.59,
            677.22,
            677.86,
            678.49,
            679.13,
            679.77,
            680.4,
            681.04,
            681.67,
            682.31,
            682.94,
            683.58,
            684.22,
            684.85,
            685.49,
            686.12,
            686.76,
            687.4,
            688.03,
            688.67,
            689.31,
            689.94,
            690.58,
            691.22,
            691.85,
            692.49,
            693.13,
            693.76,
            694.4,
            695.04,
            695.67,
            696.31,
            696.95,
            697.59,
            698.22,
            698.86,
            699.5,
            700.14,
            700.77,
            701.41,
            702.05,
            702.69,
            703.32,
            703.96,
            704.6,
            705.24,
            705.88,
            706.51,
            707.15,
            707.79,
            708.43,
            709.07,
            709.71,
            710.34,
            710.98,
            711.62,
            712.26,
            712.9,
            713.54,
            714.18,
            714.82,
            715.46,
            716.09,
            716.73,
            717.37,
            718.01,
            718.65,
            719.29,
            719.93,
            720.57,
            721.21,
            721.85,
            722.49,
            723.13,
            723.77,
            724.41,
            725.05,
            725.69,
            726.33,
            726.97,
            727.61,
            728.25,
            728.89,
            729.53,
            730.17,
            730.81,
            731.45,
            732.09,
            732.73,
            733.37,
            734.02,
            734.66,
            735.3,
            735.94,
            736.58,
            737.22,
            737.86,
            738.5,
            739.14,
            739.79,
            740.43,
            741.07,
            741.71,
            742.35,
            742.99,
            743.64,
            744.28,
            744.92,
            745.56,
            746.2,
            746.85,
            747.49,
            748.13,
            748.77,
            749.41,
            750.06,
            750.7,
            751.34,
            751.99,
            752.63,
            753.27,
            753.91,
            754.56,
            755.2,
            755.84,
            756.49,
            757.13,
            757.77,
            758.41,
            759.06,
            759.7,
            760.34,
            760.99,
            761.63,
            762.28,
            762.92,
            763.56,
            764.21,
            764.85,
            765.49,
            766.14,
            766.78,
            767.43,
            768.07,
            768.72,
            769.36,
            770.0,
            770.65,
            771.29,
            771.94,
            772.58,
            773.23,
            773.87,
            774.52,
            775.16,
            775.81,
            776.45,
            777.1,
            777.74,
            778.39,
            779.03,
            779.68,
            780.32,
            780.97,
            781.61,
            782.26,
            782.9,
            783.55,
            784.2,
            784.84,
            785.49,
            786.13,
            786.78,
            787.43,
            788.07,
            788.72,
            789.36,
            790.01,
            790.66,
            791.3,
            791.95,
            792.6,
            793.24,
            793.89,
            794.54,
            795.18,
            795.83,
            796.48,
            797.12,
            797.77,
            798.42,
            799.07,
            799.71,
            800.36,
            801.01,
            801.66,
            802.3,
            802.95,
            803.6,
            804.25,
            804.89,
            805.54,
            806.19,
            806.84,
            807.49,
            808.13,
            808.78,
            809.43,
            810.08,
            810.73,
            811.38,
            812.02,
            812.67,
            813.32,
            813.97,
            814.62,
            815.27,
            815.92,
            816.56,
            817.21,
            817.86,
            818.51,
            819.16,
            819.81,
            820.46,
            821.11,
            821.76,
            822.41,
            823.06,
            823.71,
            824.36,
            825.01,
            825.66,
            826.31,
            826.96,
            827.61,
            828.26,
            828.91,
            829.56,
            830.21,
            830.86,
            831.51,
            832.16,
            832.81,
            833.46,
            834.11,
            834.76,
            835.41,
            836.06,
            836.71,
            837.36,
            838.01,
            838.66,
            839.32,
            839.97,
            840.62,
            841.27,
            841.92,
            842.57,
            843.22,
            843.88,
            844.53,
            845.18,
            845.83,
            846.48,
            847.13,
            847.79,
            848.44,
            849.09,
            849.74,
            850.39,
            851.05,
            851.7,
            852.35,
            853.0,
            853.66,
            854.31,
            854.96,
            855.61,
            856.27,
            856.92,
            857.57,
            858.22,
            858.88,
            859.53,
            860.18,
            860.84,
            861.49,
            862.14,
            862.8,
            863.45,
            864.1,
            864.76,
            865.41,
            866.06,
            866.72,
            867.37,
            868.03,
            868.68,
            869.33,
            869.99,
            870.64,
            871.3,
            871.95,
            872.61,
            873.26,
            873.91,
            874.57,
            875.22,
            875.88,
            876.53,
            877.19,
            877.84,
            878.5,
            879.15,
            879.81,
            880.46,
            881.12,
            881.77,
            882.43,
            883.08,
            883.74,
            884.39,
            885.05,
            885.7,
            886.36,
            887.02,
            887.67,
            888.33,
            888.98,
            889.64,
            890.29,
            890.95,
            891.61,
            892.26,
            892.92,
            893.58,
            894.23,
            894.89,
            895.55,
            896.2,
            896.86,
            897.51,
            898.17,
            898.83,
            899.49,
            900.14,
            900.8,
            901.46,
            902.11,
            902.77,
            903.43,
            904.09,
            904.74,
            905.4,
            906.06,
            906.72,
            907.37,
            908.03,
            908.69,
            909.35,
            910.0,
            910.66,
            911.32,
            911.98,
            912.64,
            913.29,
            913.95,
            914.61,
            915.27,
            915.93,
            916.59,
            917.24,
            917.9,
            918.56,
            919.22,
            919.88,
            920.54,
            921.2,
            921.86,
            922.52,
            923.17,
            923.83,
            924.49,
            925.15,
            925.81,
            926.47,
            927.13,
            927.79,
            928.45,
            929.11,
            929.77,
            930.43,
            931.09,
            931.75,
            932.41,
            933.07,
            933.73,
            934.39,
            935.05,
            935.71,
            936.37,
            937.03,
            937.69,
            938.35,
            939.01,
            939.67,
            940.33,
            940.99,
            941.66,
            942.32,
            942.98,
            943.64,
            944.3,
            944.96,
            945.62,
            946.28,
            946.95,
            947.61,
            948.27,
            948.93,
            949.59,
            950.25,
            950.91,
            951.58,
            952.24,
            952.9,
            953.56,
            954.22,
            954.89,
            955.55,
            956.21,
            956.87,
            957.54,
            958.2,
            958.86,
            959.52,
            960.19,
            960.85,
            961.51,
            962.17,
            962.84,
            963.5,
            964.16,
            964.83,
            965.49,
            966.15,
            966.82,
            967.48,
            968.14,
            968.81,
            969.47,
            970.13,
            970.8,
            971.46,
            972.12,
            972.79,
            973.45,
            974.12,
            974.78,
            975.44,
            976.11,
            976.77,
            977.44,
            978.1,
            978.77,
            979.43,
            980.09,
            980.76,
            981.42,
            982.09,
            982.75,
            983.42,
            984.08,
            984.75,
            985.41,
            986.08,
            986.74,
            987.41,
            988.07,
            988.74,
            989.4,
            990.07,
            990.74,
            991.4,
            992.07,
            992.73,
            993.4,
            994.06,
            994.73,
            995.4,
            996.06,
            996.73,
            997.39,
            998.06,
            998.73,
            999.39,
            1000.21,
            1005.87,
            1011.53,
            1017.19,
            1022.85,
            1028.51,
            1034.17,
            1039.83,
            1045.48,
            1051.14,
            1056.8,
            1062.46,
            1068.12,
            1073.77,
            1079.43,
            1085.09,
            1090.75,
            1096.4,
            1102.06,
            1107.72,
            1113.37,
            1119.03,
            1124.69,
            1130.34,
            1136.0,
            1141.65,
            1147.31,
            1152.97,
            1158.62,
            1164.28,
            1169.93,
            1175.59,
            1181.24,
            1186.89,
            1192.55,
            1198.2,
            1203.86,
            1209.51,
            1215.16,
            1220.81,
            1226.47,
            1232.12,
            1237.77,
            1243.42,
            1249.08,
            1254.73,
            1260.38,
            1266.03,
            1271.68,
            1277.33,
            1282.98,
            1288.63,
            1294.28,
            1299.93,
            1305.58,
            1311.23,
            1316.88,
            1322.53,
            1328.18,
            1333.83,
            1339.47,
            1345.12,
            1350.77,
            1356.42,
            1362.06,
            1367.71,
            1373.36,
            1379.0,
            1384.65,
            1390.29,
            1395.94,
            1401.58,
            1407.23,
            1412.87,
            1418.52,
            1424.16,
            1429.8,
            1435.45,
            1441.09,
            1446.73,
            1452.37,
            1458.01,
            1463.66,
            1469.3,
            1474.94,
            1480.58,
            1486.22,
            1491.86,
            1497.5,
            1503.14,
            1508.78,
            1514.42,
            1520.05,
            1525.69,
            1531.33,
            1536.97,
            1542.6,
            1548.24,
            1553.87,
            1559.51,
            1565.15,
            1570.78,
            1576.42,
            1582.05,
            1587.68,
            1593.32,
            1598.95,
            1604.58,
            1610.22,
            1615.85,
            1621.48,
            1627.11,
            1632.74,
            1638.37,
            1644.0,
            1649.63,
            1655.26,
            1660.89,
            1666.52,
            1672.15,
            1677.77,
            1683.4,
            1689.03,
            1694.65,
            1700.28,
            1705.91,
            1711.53,
            1717.16,
            1722.78,
            1728.4,
            1734.03,
            1739.65,
            1745.27,
            1750.89,
            1756.52,
            1762.14,
            1767.76,
            1773.38,
            1779.0,
            1784.62,
            1790.24,
            1795.85,
            1801.47,
            1807.09,
            1812.71,
            1818.32,
            1823.94,
            1829.56,
            1835.17,
            1840.79,
            1846.4,
            1852.01,
            1857.63,
            1863.24,
            1868.85,
            1874.46,
            1880.07,
            1885.68,
            1891.3,
            1896.9,
            1902.51,
            1908.12,
            1913.73,
            1919.34,
            1924.95,
            1930.55,
            1936.16,
            1941.76,
            1947.37,
            1952.97,
            1958.58,
            1964.18,
            1969.78,
            1975.39,
            1980.99,
            1986.59,
            1992.19,
            1997.79,
            2003.39,
            2008.99,
            2014.59,
            2020.19,
            2025.78,
            2031.38,
            2036.98,
            2042.57,
            2048.17,
            2053.76,
            2059.36,
            2064.95,
            2070.54,
            2076.14,
            2081.73,
            2087.32,
            2092.91,
            2098.5,
            2104.09,
            2109.68,
            2115.27,
            2120.85,
            2126.44,
            2132.03,
            2137.61,
            2143.2,
            2148.78,
            2154.37,
            2159.95,
            2165.53,
            2171.11,
            2176.7,
            2182.28,
            2187.86,
            2193.44,
            2199.01,
            2204.59,
            2210.17,
            2215.75,
            2221.32,
            2226.9,
            2232.47,
            2238.05,
            2243.62,
            2249.2,
            2254.77,
            2260.34,
            2265.91,
            2271.48,
            2277.05,
            2282.62,
            2288.19,
            2293.76,
            2299.32,
            2304.89,
            2310.46,
            2316.02,
            2321.58,
            2327.15,
            2332.71,
            2338.27,
            2343.84,
            2349.4,
            2354.96,
            2360.52,
            2366.07,
            2371.63,
            2377.19,
            2382.75,
            2388.3,
            2393.86,
            2399.41,
            2404.97,
            2410.52,
            2416.07,
            2421.62,
            2427.17,
            2432.72,
            2438.27,
            2443.82,
            2449.37,
            2454.92,
            2460.46,
            2466.01,
            2471.55,
            2477.1,
            2482.64,
            2488.18,
            2493.72,
            2499.26
            ]

In [ ]:
numspectbydoc = 16
numdocbydatafile = 10

numberskippedbands = 2
# Discretize data
numdiscretizedintensities = 64

binitial = True

fnames = []
docs1 = []
docs2 = []
docs3 = []
docs4 = []

for idxfile in range(len(files_x)):
    # Load X data with shape [num_samples, num_spectra].
    filenpz = np.load(folder_db + files_x[idxfile])
    X_data = filenpz['X']
    filenpz.close()

    # Normalize to [0, 1] using the global range.
    X_data = (X_data.astype('float') - min_db) / global_range

    # Number of spectra in the data file.
    numSpectra = X_data.shape[0]
    # Number of bands per spectrum.
    dimSpectra = X_data.shape[1]

    for idxd in range(numdocbydatafile):
        fnames.append(files_x[idxfile])

        # Create documents.
        idx_selectedSpectra = np.random.choice(numSpectra, numspectbydoc)

        my_waves = wavelength[::numberskippedbands]
        X_currentdoc = np.squeeze(X_data[idx_selectedSpectra, ::numberskippedbands])
        X_curr_dis = (X_currentdoc * (numdiscretizedintensities - 1)).astype('int')
        # X_currentdoc = np.reshape(X_currentdoc, (numdocbydatafile, numspectbydoc * dimSpectra))

        # Case 1: N spectra normalized with M intensity levels.
        # Words are wavelengths.
        # For each document, the occurrence count of each wavelength
        # is the summed discretized intensity across the selected spectra.
        sum_waves = np.sum(X_curr_dis, axis=0)
        nested_l = [[str(pair[0]).zfill(4) + 'nm'] * pair[1] for pair in zip(my_waves, sum_waves) if pair[1] > 0]
        flat_list = [item for sublist in nested_l for item in sublist]

        docs1.append(flat_list)


In [ ]:
flat_list

In [ ]:
d1 = {'document': docs1, 'filename': fnames}
df_1 = pd.DataFrame(data=d1)

d2 = {'document': docs2, 'filename': fnames}
df_2 = pd.DataFrame(data=d2)

d3 = {'document': docs3, 'filename': fnames}
df_3 = pd.DataFrame(data=d3)

In [ ]:
df_3

# Option A

Use each band in the document as an element of the alphabet.
## The intensity becomes the occurrence count after discretization


In [ ]:
# Create documents from the available data

numspectbydoc = 16
numdocbydatafile = 10

binitial = True

X_rawdocuments = []
for idxfile in range(len(files_x)):
  # Load X data with shape [num_samples, num_spectra].
  filenpz = np.load(folder_db + files_x[idxfile])
  X_data = filenpz['X']
  filenpz.close()

  # Number of spectra in the data file.
  numSpectra = X_data.shape[0]
  # Number of bands per spectrum.
  dimSpectra = X_data.shape[1]

  # Create documents.
  idx_selectedSpectra = np.random.choice(numSpectra, numspectbydoc * numdocbydatafile)
  X_currentdoc = np.squeeze(X_data[idx_selectedSpectra, :])
  X_currentdoc = np.reshape(X_currentdoc, (numdocbydatafile, numspectbydoc * dimSpectra))

  if binitial:
    binitial = False

    X_rawdocuments = X_currentdoc
  else:
    X_rawdocuments = np.append(X_rawdocuments, X_currentdoc, axis=0)

X_rawdocuments.shape


In [ ]:
# Discretize the documents
numdiscretizedintensities = 256
X_rawdocuments_dis = ((numdiscretizedintensities - 1) * X_rawdocuments.astype('float') / max(np.ravel(X_rawdocuments.astype('float')))).astype('int')


In [ ]:
# Documents in Dirichlet-ready format
N_DV_HS1 = X_rawdocuments_dis


In [ ]:
X_rawdocuments_dis.shape

# Option B

Use the discretized intensity as the vocabulary, then count how many
bands fall into each intensity level.


In [ ]:
# Create documents from the available data

numspectbydoc = 16
numdocbydatafile = 100

binitial = True

X_rawdocuments = []
for idxfile in range(len(files_x)):
  # Load X data with shape [num_samples, num_spectra].
  filenpz = np.load(folder_db + files_x[idxfile])
  X_data = filenpz['X']
  filenpz.close()

  # Number of spectra in the data file.
  numSpectra = X_data.shape[0]
  # Number of bands per spectrum.
  dimSpectra = X_data.shape[1]

  # Create documents.
  idx_selectedSpectra = np.random.choice(numSpectra, numspectbydoc * numdocbydatafile)
  X_currentdoc = np.squeeze(X_data[idx_selectedSpectra, :])
  X_currentdoc = np.reshape(X_currentdoc, (numdocbydatafile, numspectbydoc * dimSpectra))

  if binitial:
    binitial = False

    X_rawdocuments = X_currentdoc
  else:
    X_rawdocuments = np.append(X_rawdocuments, X_currentdoc, axis=0)

X_rawdocuments.shape

D = X_rawdocuments.shape[0]


In [ ]:
# Discretize the documents
numdiscretizedintensities = 256
X_rawdocuments_dis = ((numdiscretizedintensities - 1) * X_rawdocuments.astype('float') / max(np.ravel(X_rawdocuments.astype('float')))).astype('int')

N_DV_HS2 = np.zeros((D, numdiscretizedintensities)).astype('int')

# Count the occurrences of each intensity token.
for idxd in range(D):
    N_DV_HS2[idxd, :] = np.bincount(X_rawdocuments_dis[idxd, :], minlength=numdiscretizedintensities)


In [ ]:
for idxd in range(D):
    plt.plot(X_rawdocuments[idxd,:])

# Option C

Use each band in the document as an element of the alphabet.
## The intensity becomes the occurrence count after discretization

## Reduced-band version


In [ ]:
# Create documents from the available data with a reduced number of bands

numspectbydoc = 16
numdocbydatafile = 100
numberskippedbands = 5

binitial = True

X_rawdocuments = []
for idxfile in range(len(files_x)):
  filenpz = np.load(folder_db + files_x[idxfile])
  X_data = filenpz['X'][:, ::numberskippedbands]
  filenpz.close()

  # Number of spectra in the data file.
  numSpectra = X_data.shape[0]
  # Number of bands per spectrum.
  dimSpectra = X_data.shape[1]

  # Create documents.
  idx_selectedSpectra = np.random.choice(numSpectra, numspectbydoc * numdocbydatafile)
  X_currentdoc = np.squeeze(X_data[idx_selectedSpectra, :])
  X_currentdoc = np.reshape(X_currentdoc, (numdocbydatafile, numspectbydoc * dimSpectra))

  if binitial:
    binitial = False

    X_rawdocuments = X_currentdoc
  else:
    X_rawdocuments = np.append(X_rawdocuments, X_currentdoc, axis=0)

X_rawdocuments.shape


In [ ]:
# Discretize the documents
numdiscretizedintensities = 256
X_rawdocuments_dis = ((numdiscretizedintensities - 1) * X_rawdocuments.astype('float') / max(np.ravel(X_rawdocuments.astype('float')))).astype('int')


In [ ]:
X_rawdocuments_dis.shape

In [ ]:
N_DV_HS1.shape[0]



In [ ]:
# Documents in Dirichlet-ready format
N_DV_HS3 = X_rawdocuments_dis


In [ ]:
N_DV_HS1.shape

for idx in range(N_DV_HS1.shape[0]):
    plt.plot(np.ravel(N_DV_HS1[idx,:]))

In [ ]:
N_DV_HS2.shape

for idx in range(N_DV_HS2.shape[0]):
    plt.plot(np.ravel(N_DV_HS2[idx,:]))

plt.xlim(0 , 64)

In [ ]:
N_DV_HS3.shape
for idx in range(N_DV_HS3.shape[0]):
    plt.plot(np.ravel(N_DV_HS3[idx,:]))

In [ ]:
print(max(np.ravel(N_DV_HS1)))
print(max(np.ravel(N_DV_HS2)))
print(max(np.ravel(N_DV_HS3)))

In [ ]:
N_DV_HS1.shape

In [ ]:
df_1

In [ ]:
tokenizer = ToktokTokenizer() 

df = df_1
# Clean unwanted symbols.
# df["Tokens"] = df.document.apply(clean_text)
df["Tokens"] = df.document

# Create tokens.
# df["Tokens"] = df.Tokens.apply(tokenizer.tokenize)

# Remove stopwords and standalone numbers.
# df["Tokens"] = df.Tokens.apply(filter_stopwords_and_digits)

# Apply stemming if needed.
# if use_stemmer:
#     df["Tokens"] = df.Tokens.apply(stem_words)

# df["TokensNgrams"] = [generate_ngrams(words=sentence, words_to_combine=words_per_ngram) for sentence in df.Tokens]

df.head()


In [ ]:
use_ngrams = False
if use_ngrams:
    dictionary = Dictionary(df.TokensNgrams)
else:
    dictionary = Dictionary(df.Tokens)
print(f'Vocabulary size: {len(dictionary)}')


In [ ]:
dictionary.filter_extremes(no_below=2, no_above=0.8)
print(f'Vocabulary size after filtering: {len(dictionary)}')


In [ ]:
# Create the corpus.
if use_ngrams:
    corpus = [dictionary.doc2bow(document_tokens) for document_tokens in df.TokensNgrams]
else:
    corpus = [dictionary.doc2bow(document_tokens) for document_tokens in df.Tokens]

# Show a few bag-of-words documents.
print(corpus[:7])


In [ ]:
num_topics = 10
lda = LdaModel(corpus=corpus, id2word=dictionary, 
               num_topics=num_topics, random_state=42, 
               chunksize=1000, passes=10, alpha='auto')


In [ ]:
topics = lda.print_topics(num_words=5, num_topics=20)
for topic in topics:
    print(topic)


In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pickle 
import pyLDAvis
# Visualize the topics.
pyLDAvis.enable_notebook()
ldavis_data_path = os.path.join('./ldavis_prepared_' + str(num_topics))
# This step can take some time.
# Set the condition to True when preparing the visualization locally.
if True:
    LDAvis_prepared = gensimvis.prepare(lda, corpus, dictionary)
    with open(ldavis_data_path, 'wb') as f:
        pickle.dump(LDAvis_prepared, f)
# Load the prepared pyLDAvis data from disk.
with open(ldavis_data_path, 'rb') as f:
    LDAvis_prepared = pickle.load(f)
pyLDAvis.save_html(LDAvis_prepared, './ldavis_prepared_' + str(num_topics) + '.html')
LDAvis_prepared
